# 1D Triple-Well Potential: Symmetric Configurations

Implements Section 3.3.1–3.3.2 of the thesis. Covers the effect of noise intensity on convergence and transfer learning between symmetric configurations.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

sys.path.append(os.path.abspath('..'))

# Import our custom modules
from src.problems.triple_well import TripleWellProblem
from src.models import FPNet
from src.training import train_model
from src.utils import acc_L2
from src.losses import fp_loss

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Potential Structure and Parameter Selection

Analytical characterisation of the triple-well potential $(a=0.2,\,b=1.5,\,c=3.0,\,d=0)$ and exact stationary PDFs for $\sigma \in \{0.5, 0.7, 1.0, 1.5, 2.0, 3.0\}$.

In [ ]:

a, b, c = 0.2, 1.5, 3.0
x_plot = np.linspace(-3.5, 3.5, 1000)

# Finding potential minima analytically
# V'(x) = 6ax^5 - 4bx^3 + 2cx = x*(6ax^4 - 4bx^2 + 2c) = 0
# Solving the quadratic equation for u = x^2:
# 1.2u^2 - 6u + 6 = 0
u1 = (6 + np.sqrt(36 - 4*1.2*6)) / (2*1.2)
u2 = (6 - np.sqrt(36 - 4*1.2*6)) / (2*1.2)
x_side_min  = np.sqrt(u1)   # ≈ 1.90 — side minima
x_inner_max = np.sqrt(u2)   # ≈ 1.18 — inner maxima (barriers)

def V(x):
    return a*x**6 - b*x**4 + c*x**2

V_center   = V(0)
V_side     = V(x_side_min)
V_barrier  = V(x_inner_max)

print("Potential Structure (d=0):")
print(f"  Central minimum:    x=0,      V={V_center:.4f}")
print(f"  Side minima:        x=±{x_side_min:.3f}, V={V_side:.4f}")
print(f"  Inner barriers:     x=±{x_inner_max:.3f}, V={V_barrier:.4f}")
print(f"  Barrier height (from center): ΔV={V_barrier - V_center:.4f}")
print(f"  Barrier height (from sides):  ΔV={V_barrier - V_side:.4f}")

# --- Analytical solutions for different σ ---

SIGMAS = [0.5, 0.7, 1.0, 1.5, 2.0, 3.0]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

chosen = {}  # Save for selection

for ax, sigma in zip(axes, SIGMAS):
    prob = TripleWellProblem(a=a, b=b, c=c, d=0.0, sigma=sigma)
    p = prob.exact_solution(x_plot)

    # Peak heights in the three wells
    # Center: x≈0, Sides: x≈±1.90
    idx_center = np.argmin(np.abs(x_plot - 0))
    idx_side   = np.argmin(np.abs(x_plot - x_side_min))
    h_center = p[idx_center]
    h_side   = p[idx_side]

    max_h = max(h_center, h_side)
    ratio = min(h_center, h_side) / max_h if max_h > 1e-8 else 0.0

    chosen[sigma] = {
        'h_center': h_center,
        'h_side': h_side,
        'ratio': ratio,
    }

    ax.plot(x_plot, p, 'b-', lw=2)
    ax.axvline(0,            color='gray', ls=':', lw=1, alpha=0.5)
    ax.axvline( x_side_min,  color='gray', ls=':', lw=1, alpha=0.5)
    ax.axvline(-x_side_min,  color='gray', ls=':', lw=1, alpha=0.5)
    ax.set_title(
        f'σ={sigma}\n'
        f'center={h_center:.3f}, side={h_side:.3f}, ratio={ratio:.2f}',
        fontsize=10,
    )
    ax.set_xlabel('x')
    ax.set_ylabel('p(x)')
    ax.grid(True, alpha=0.3)

plt.suptitle(
    'Triple-Well: Exact stationary PDF for different σ\n'
    f'a={a}, b={b}, c={c}, d=0 | ratio = min(peak)/max(peak)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig('triple_well_sigma_selection.png', dpi=100, bbox_inches='tight')
plt.show()

# --- Table for selecting σ ---
print(f"\n{'σ':>5} | {'h_center':>9} | {'h_side':>9} | {'ratio':>7} | Difficulty")
print("-" * 65)
for sigma, info in chosen.items():
    r = info['ratio']
    if r > 0.7:
        difficulty = "easy    (peaks similar)"
    elif r > 0.3:
        difficulty = "medium"
    else:
        difficulty = "hard    (strong contrast)"
    print(f"{sigma:>5} | {info['h_center']:>9.4f} | {info['h_side']:>9.4f} | "
          f"{r:>7.3f} | {difficulty}")

## 3.3.1 Effect of Noise Intensity on Convergence

Trains the standard PINN (no normalisation) and DL-FP across 8 seeds for $\sigma \in \{2.0,\,1.5,\,1.0\}$. Reproduces Figures 3.6 and 3.7.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.training import train_model
from src.utils import acc_L2
from src.problems.triple_well import TripleWellProblem

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_eval = np.linspace(-4.0, 4.0, 801)
SIGMAS = [2.0, 1.5, 1.0]
SEEDS  = [0, 1, 7, 13, 42, 99, 123, 256]

# ─── Part 1: Baseline PINN (no normalization) — one σ is enough ──────────────
# Show for σ=1.5 (medium difficulty, three peaks visible)

print("=== BASELINE PINN (no normalization, expected: p≡0) ===")
problem_baseline = TripleWellProblem(sigma=1.5, domain=(-4.0, 4.0))
p_exact_15 = problem_baseline.exact_solution(x_eval)

result_baseline = train_model(
    problem_baseline,
    model_config={'output_transform': 'softplus'},
    optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
    dx=0.01,
    use_normalization=False,   # ← PINN without normalization (E2)
    penalty_factors={'a1': 1.0, 'a2': 0.0, 'a3': 1.0},
    seed=42,
    device=device,
    print_every=10000,
)

result_baseline_model = result_baseline['model']
result_baseline_model.eval()

# Move evaluation tensor to the correct device
x_t = torch.tensor(x_eval, dtype=torch.float32, device=device).view(-1, 1)

with torch.no_grad():
    p_baseline = result_baseline_model(x_t).cpu().numpy().flatten()

acc_baseline = acc_L2(p_baseline, p_exact_15)
print(f"Baseline PINN acc={acc_baseline:.4f}, max(p)={p_baseline.max():.6f}")

# ─── Part 2: DL-FP training for all σ × seeds ────────────────────────────────

all_results = {}

for sigma in SIGMAS:
    print(f"\n{'='*60}\nσ = {sigma}\n{'='*60}")
    problem = TripleWellProblem(sigma=sigma, domain=(-4.0, 4.0))
    p_exact = problem.exact_solution(x_eval)
    all_results[sigma] = {}

    for seed in SEEDS:
        print(f"  Seed={seed}...")
        result = train_model(
            problem,
            model_config={'output_transform': 'softplus'},
            optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
            dx=0.01,
            use_normalization=True,
            penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
            seed=seed,
            device=device,
            print_every=99999, # Print only final line
        )

        model = result['model']
        model.eval()
        with torch.no_grad():
            p_pred = model(x_t).cpu().numpy().flatten()

        acc = acc_L2(p_pred, p_exact)

        n = len(x_eval)
        third = n // 3
        h_left   = p_pred[:third].max()
        h_center = p_pred[third:2*third].max()
        h_right  = p_pred[2*third:].max()
        h_max    = max(h_left, h_center, h_right)
        h_min    = min(h_left, h_center, h_right)
        ratio    = h_min / h_max if h_max > 1e-6 else 0.0

        peaks_found = sum([
            h_left   > 0.2 * h_max,
            h_center > 0.2 * h_max,
            h_right  > 0.2 * h_max,
        ])

        all_results[sigma][seed] = {
            'acc': acc, 'ratio': ratio,
            'peaks_found': peaks_found, 'p': p_pred,
            'h': (h_left, h_center, h_right),
            'history': result['history'],
        }
        print(f"  -> Acc={acc:.4f} | Peaks={peaks_found}/3 | "
              f"L={h_left:.3f} C={h_center:.3f} R={h_right:.3f}")

# ─── Summary ─────────────────────────────────────────────────────────────────

print(f"\n{'='*75}")
print("SUMMARY: Triple-Well | Adam 30000 | softplus")
print(f"{'='*75}")
print(f"{'Seed':>6} | " + " | ".join(f"  σ={s} (Acc/Peaks)  " for s in SIGMAS))
print("-" * 75)
for seed in SEEDS:
    parts = []
    for sigma in SIGMAS:
        r = all_results[sigma][seed]
        parts.append(f"Acc={r['acc']:.3f} / {r['peaks_found']}pk")
    print(f"{seed:>6} | " + " | ".join(f"{p:>22}" for p in parts))
print("-" * 75)
print("Success rate (3 peaks found):")
for sigma in SIGMAS:
    n_success = sum(
        1 for seed in SEEDS
        if all_results[sigma][seed]['peaks_found'] == 3
    )
    avg_acc = np.mean([all_results[sigma][seed]['acc'] for seed in SEEDS])
    print(f"  σ={sigma:<3}: {n_success}/{len(SEEDS)} | Mean Acc={avg_acc:.4f}")

# ─── Figure 1: Baseline PINN vs DL-FP ────────────────────────────────────────
# Show for σ=1.5: exact / baseline(p≡0) / DL-FP best / DL-FP worst

best_seed_15  = max(SEEDS, key=lambda s: all_results[1.5][s]['acc'])
worst_seed_15 = min(SEEDS, key=lambda s: all_results[1.5][s]['acc'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PINN vs DL-FP
ax = axes[0]
ax.plot(x_eval, p_exact_15, 'k-', lw=2, label='Exact solution')
ax.plot(x_eval, p_baseline, 'b:', lw=2,
        label=f'PINN (no norm) acc={acc_baseline:.4f}')
ax.plot(x_eval, all_results[1.5][best_seed_15]['p'], 'r--', lw=2,
        label=f"DL-FP best seed={best_seed_15} "
              f"acc={all_results[1.5][best_seed_15]['acc']:.4f}")
ax.set_title('PINN (no normalization) vs DL-FP\nσ=1.5', fontsize=11)
ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Right: Loss convergence for baseline PINN
ax = axes[1]
h = result_baseline['history']
epochs = range(len(h['total']))
ax.semilogy(epochs, h['total'], 'k-',  lw=1.5, label='Total Loss')
ax.semilogy(epochs, h['pde'],   'b--', lw=1.5, label='PDE Loss')
ax.semilogy(epochs, h['bound'], 'g--', lw=1.5, label='Boundary Loss')
ax.set_title('PINN without normalization: Loss convergence\n'
             '(converges to trivial solution p≡0)', fontsize=11)
ax.set_xlabel('Epoch'); ax.set_ylabel('Log(Loss)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle('Experiment 5a: Why normalization is necessary', fontsize=14)
plt.tight_layout()
plt.savefig('exp5a_baseline_pinn.png', dpi=100, bbox_inches='tight')
plt.show()

# ─── Figure 2: DL-FP results across σ ────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, sigma in zip(axes, SIGMAS):
    problem = TripleWellProblem(sigma=sigma, domain=(-4.0, 4.0))
    p_exact = problem.exact_solution(x_eval)

    best_seed  = max(SEEDS, key=lambda s: all_results[sigma][s]['acc'])
    worst_seed = min(SEEDS, key=lambda s: all_results[sigma][s]['acc'])

    ax.plot(x_eval, p_exact, 'k-', lw=2, label='Exact')
    ax.plot(x_eval, all_results[sigma][best_seed]['p'], 'r--', lw=2,
            label=f"Best seed={best_seed}\n"
                  f"acc={all_results[sigma][best_seed]['acc']:.4f}, "
                  f"{all_results[sigma][best_seed]['peaks_found']}pk")
    ax.plot(x_eval, all_results[sigma][worst_seed]['p'], 'b:', lw=1.5, alpha=0.8,
            label=f"Worst seed={worst_seed}\n"
                  f"acc={all_results[sigma][worst_seed]['acc']:.4f}, "
                  f"{all_results[sigma][worst_seed]['peaks_found']}pk")
    ax.set_title(f'σ={sigma}', fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('p(x)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(
    'Experiment 5b: DL-FP Triple-Well | Adam 30000 | Best vs Worst seed',
    fontsize=14, y=1.02
)
plt.tight_layout()
plt.savefig('exp5b_triple_well_sigma.png', dpi=100, bbox_inches='tight')
plt.show()

# ─── Figure 3: Loss convergence — best run per σ ─────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, sigma in zip(axes, SIGMAS):
    best_seed = max(SEEDS, key=lambda s: all_results[sigma][s]['acc'])
    h = all_results[sigma][best_seed]['history']
    epochs = range(len(h['total']))

    ax.semilogy(epochs, h['total'], 'k-',  lw=1.5, label='Total')
    ax.semilogy(epochs, h['pde'],   'b--', lw=1.5, label='PDE')
    ax.semilogy(epochs, h['norm'],  'r--', lw=1.5, label='Norm')
    ax.semilogy(epochs, h['bound'], 'g--', lw=1.5, label='Boundary')

    final_pde = h['pde'][-1]
    ax.set_title(
        f'σ={sigma} | Seed={best_seed}\n'
        f'Acc={all_results[sigma][best_seed]["acc"]:.4f} | '
        f'Final PDE Loss={final_pde:.2e}',
        fontsize=10,
    )
    ax.set_xlabel('Epoch'); ax.set_ylabel('Log(Loss)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(
    'Experiment 5c: Loss convergence — Best seed per σ (DL-FP)',
    fontsize=14, y=1.02
)
plt.tight_layout()
plt.savefig('exp5c_loss_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Map all_results to the format expected by the transfer learning experiment below
trained = {
    (sigma, seed, True): all_results[sigma][seed]
    for sigma in SIGMAS
    for seed in SEEDS
    if seed in all_results.get(sigma, {})
}


## 3.3.2 Transfer Learning — Symmetric Configurations

Trains source models at $\sigma \in \{2.0, 1.5\}$ and fine-tunes on $\sigma \in \{1.5, 1.0\}$. Reproduces Figures 3.8, 3.9, 3.10.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.training import train_model
from src.utils import acc_L2

x_eval = np.linspace(-4.0, 4.0, 801)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x_t = torch.tensor(x_eval, dtype=torch.float32, device=device).view(-1, 1)

# ─── Step 1: Retrain source models (2 runs) ──────────────────────────────────

source_models = {}

for sigma, seed in [(2.0, 256), (1.5, 256)]:
    print(f"\n[SOURCE] σ={sigma}, seed={seed}")
    problem = TripleWellProblem(sigma=sigma, domain=(-4.0, 4.0))
    result = train_model(
        problem,
        model_config={'output_transform': 'softplus'},
        optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
        dx=0.01,
        penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
        seed=seed,
        device=device,
        print_every=99999,
    )
    source_models[sigma] = result['model']
    source_models[sigma].eval()

    with torch.no_grad():
        p_pred = source_models[sigma](x_t).cpu().numpy().flatten()
    acc = acc_L2(p_pred, problem.exact_solution(x_eval))
    print(f"  -> acc={acc:.4f} (expected 0.99+)")

# ─── Step 2: TL runs ──────────────────────────────────────────────────────────

# Three pairs: (source_sigma, target_sigma)
TL_PAIRS = [
    (2.0, 1.5), # Small leap
    (2.0, 1.0), # Large leap
    (1.5, 1.0), # Small leap from a harder baseline
]

tl_results = {}

for src_sigma, tgt_sigma in TL_PAIRS:
    print(f"\n[TL] Source σ={src_sigma} → Target σ={tgt_sigma}")
    problem_tgt = TripleWellProblem(sigma=tgt_sigma, domain=(-4.0, 4.0))
    p_exact_tgt = problem_tgt.exact_solution(x_eval)

    result = train_model(
        problem_tgt,
        model_config={'output_transform': 'softplus'},
        optimizer_config={'type': 'adam', 'epochs': 30000, 'lr': 1e-3},
        dx=0.01,
        use_normalization=True,
        penalty_factors={'a1': 1.0, 'a2': 1.0, 'a3': 1.0},
        seed=0,  # seed doesn't matter much after load_state_dict
        device=device,
        init_model=source_models[src_sigma],
        print_every=99999,
    )

    model = result['model']
    model.eval()
    with torch.no_grad():
        p_tl = model(x_t).cpu().numpy().flatten()

    acc = acc_L2(p_tl, p_exact_tgt)
    print(f"  -> acc={acc:.4f}")

    tl_results[(src_sigma, tgt_sigma)] = {
        'p': p_tl,
        'acc': acc,
        'history': result['history'],
    }

# ─── Step 3: Baseline "without TL" for each target ───────────────────────────
# Pulling from the previously populated 'trained' dictionary (Experiment 5)

# target σ=1.5: seed=42, acc=0.3355 (localization to central well)
# target σ=1.0: seed=42, acc=0.6724 (localization to central well)
BASELINES = {
    1.5: {
        'p':    trained[(1.5, 42, True)]['p'],
        'acc':  trained[(1.5, 42, True)]['acc'],
        'seed': 42,
    },
    1.0: {
        'p':    trained[(1.0, 42, True)]['p'],
        'acc':  trained[(1.0, 42, True)]['acc'],
        'seed': 42,
    },
}

# ─── Step 4: Figures ──────────────────────────────────────────────────────────

for src_sigma, tgt_sigma in TL_PAIRS:
    problem_tgt = TripleWellProblem(sigma=tgt_sigma, domain=(-4.0, 4.0))
    p_exact     = problem_tgt.exact_solution(x_eval)
    baseline    = BASELINES[tgt_sigma]
    tl          = tl_results[(src_sigma, tgt_sigma)]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # --- Cell 1: Without TL ---
    ax = axes[0]
    ax.plot(x_eval, p_exact, 'k-', lw=2, label='Exact solution')
    ax.plot(x_eval, baseline['p'], color='#e74c3c', ls='--', lw=2,
            label=f"Without TL (seed={baseline['seed']})\n"
                  f"acc={baseline['acc']:.4f}")
    ax.set_title(f'Without Transfer Learning\nTarget: σ={tgt_sigma}',
                 fontsize=12)
    ax.set_xlabel('x', fontsize=11); ax.set_ylabel('p(x)', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # --- Cell 2: With TL ---
    ax = axes[1]
    ax.plot(x_eval, p_exact, 'k-', lw=2, label='Exact solution')
    ax.plot(x_eval, tl['p'], color='#27ae60', ls='--', lw=2,
            label=f"With Transfer Learning\nacc={tl['acc']:.4f}")
    ax.set_title(f'With Transfer Learning\n'
                 f'Source: σ={src_sigma} → Target: σ={tgt_sigma}',
                 fontsize=12)
    ax.set_xlabel('x', fontsize=11); ax.set_ylabel('p(x)', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # --- Cell 3: Loss convergence (TL run) ---
    ax = axes[2]
    h  = tl['history']
    ep = range(len(h['total']))
    ax.semilogy(ep, h['total'], 'k-',  lw=1.5, label='Total')
    ax.semilogy(ep, h['pde'],   'b--', lw=1.5, label='PDE')
    ax.semilogy(ep, h['norm'],  'r--', lw=1.5, label='Norm')
    ax.semilogy(ep, h['bound'], 'g--', lw=1.5, label='Boundary')
    ax.axhline(h['pde'][-1], color='blue', ls=':', lw=1, alpha=0.5)
    ax.set_title(f'Loss convergence (TL run)\n'
                 f'Final PDE loss = {h["pde"][-1]:.2e}',
                 fontsize=12)
    ax.set_xlabel('Epoch', fontsize=11); ax.set_ylabel('Log(Loss)', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.suptitle(
        f'Experiment 7: Transfer Learning | '
        f'Source σ={src_sigma} → Target σ={tgt_sigma}',
        fontsize=14, y=1.02
    )
    plt.tight_layout()

    # Safe filename generation
    fname = f'exp7_TL_src{str(src_sigma).replace(".", "")}_tgt{str(tgt_sigma).replace(".", "")}.png'
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

# ─── Summary ─────────────────────────────────────────────────────────────────

print(f"\n{'='*55}")
print("SUMMARY: Transfer Learning — Triple-Well")
print(f"{'='*55}")
print(f"{'Pair':>20} | {'Without TL':>12} | {'With TL':>10}")
print("-" * 55)
for src_sigma, tgt_sigma in TL_PAIRS:
    bl  = BASELINES[tgt_sigma]
    tl  = tl_results[(src_sigma, tgt_sigma)]
    print(f"σ={src_sigma} → σ={tgt_sigma}:  "
          f"  acc={bl['acc']:.4f}  →  acc={tl['acc']:.4f}")